# Lab 10: The Inverse Problem — Building Intuition

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/11_inverse_problem_lab.ipynb)

## Introduction

In Lab 10 you built a Green's function matrix $\mathbf{G}$ and used it to predict surface
displacements from a known slip distribution — the **forward problem**. In this lab you run
that in reverse: you start from the same synthetic fault, generate noisy "observed" data by
forward modeling, and then try to recover the original slip distribution — the **inverse problem**.

Because you know the true answer, you can directly measure how well you recover it.
This lets you build concrete intuition for concepts that are easy to state abstractly but hard
to feel: noise amplification, the role of regularization, the meaning of the L-curve corner,
and what the resolution matrix actually tells you about a slip inversion.

## Learning objectives

By the end, you will be able to:

- explain why adding noise to the forward problem makes the inverse problem ill-posed
- demonstrate that direct least-squares inversion amplifies noise catastrophically
- apply damping and smoothing regularization and describe what each sacrifices
- explain why the L-curve has a corner only when noise is present
- compute the resolution matrix and interpret it as the blurring function of the inversion
- connect the resolution kernel width to the discrepancy between the true and recovered slip

## Notebook outline
- [Setup](#setup)
- [Part I: The forward model and noise](#part-i-the-forward-model-and-noise)
- [Part II: Damping regularization](#part-ii-damping-regularization)
- [Part III: Smoothing regularization](#part-iii-smoothing-regularization)
- [Part IV: The L-curve](#part-iv-the-l-curve)
- [Part V: Resolution matrix](#part-v-the-resolution-matrix)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Setup


In [ ]:
%pip install -q cutde

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cutde.halfspace as hs
from scipy.linalg import lstsq

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.titlesize': 11})

NU = 0.25   # Poisson's ratio

# ── Fault geometry helpers (same as Lab 10) ───────────────────────────────
def fault_unit_vectors(strike_deg, dip_deg):
    s = np.radians(strike_deg); d = np.radians(dip_deg)
    along   = np.array([np.sin(s),  np.cos(s), 0.])
    downdip = np.array([np.cos(d)*np.cos(s + np.pi/2),
                       -np.cos(d)*np.sin(s + np.pi/2), -np.sin(d)])
    return along, downdip


def make_fault_patches(n_along, n_down,
                        strike=0, dip=90, rake=180,
                        length_km=40, width_km=15, depth_top_km=2):
    """
    Divide a rectangular fault into n_along × n_down patches.
    Returns triangle array (2*N_patches, 3, 3) and patch centres (N_patches, 3).
    """
    along, downdip = fault_unit_vectors(strike, dip)
    r  = np.radians(rake)
    C0 = np.array([0., 0., -depth_top_km * 1e3])
    dl = length_km * 1e3 / n_along
    dw = width_km  * 1e3 / n_down
    tris, cents = [], []
    for i in range(n_along):
        for j in range(n_down):
            corner = C0 + i*dl*along + j*dw*downdip
            P = [corner,
                 corner + dl*along,
                 corner + dl*along + dw*downdip,
                 corner +            dw*downdip]
            tris += [[P[0],P[1],P[2]], [P[0],P[2],P[3]]]
            cents.append(corner + 0.5*dl*along + 0.5*dw*downdip)
    return np.array(tris, dtype=float), np.array(cents)


def build_laplacian(n_along, n_down):
    """
    Finite-difference Laplacian operator for an n_along x n_down patch grid.
    L[k,k] = number of neighbours; L[k,n] = -1 for each neighbour n.
    Minimising ||L m||^2 penalises spatial roughness.
    """
    N = n_along * n_down
    L = np.zeros((N, N))
    def idx(i, j): return i * n_down + j
    for i in range(n_along):
        for j in range(n_down):
            k = idx(i, j)
            nbrs = []
            if i > 0:          nbrs.append(idx(i-1, j))
            if i < n_along-1:  nbrs.append(idx(i+1, j))
            if j > 0:          nbrs.append(idx(i, j-1))
            if j < n_down-1:   nbrs.append(idx(i, j+1))
            L[k, k] = len(nbrs)
            for n in nbrs: L[k, n] = -1
    return L


print('Setup complete.')


## Part I: The forward model and noise

### 1a — Build the fault and the true slip distribution

We use the same vertical right-lateral strike-slip fault from Lab 10:
strike 0°, dip 90°, rake 180°, 40 km long, 15 km wide, top at 2 km depth.
We divide it into a **5 × 4 grid of 20 patches**.

The "true" slip distribution is a **Gaussian asperity** centred on the fault —
the kind of smooth, concentrated slip pattern that real earthquakes produce.


In [ ]:
# Fault discretisation
N_ALONG, N_DOWN = 5, 4
N_PATCHES = N_ALONG * N_DOWN

tris_all, cents = make_fault_patches(N_ALONG, N_DOWN)
L_lap = build_laplacian(N_ALONG, N_DOWN)

# True slip: Gaussian asperity centred on the fault (m)
I, J    = np.meshgrid(np.arange(N_ALONG), np.arange(N_DOWN), indexing='ij')
ci, cj  = N_ALONG/2 - 0.5, N_DOWN/2 - 0.5
m_true  = 3.0 * np.exp(-((I - ci)**2 / 1.5 + (J - cj)**2 / 0.8)).ravel()

print(f"Fault: {N_PATCHES} patches ({N_ALONG} along-strike × {N_DOWN} down-dip)")
print(f"True slip: min={m_true.min():.2f} m,  max={m_true.max():.2f} m")

# Visualise the true slip on the fault
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                   m_true.reshape(N_ALONG, N_DOWN).T,
                   cmap='hot_r', vmin=0, vmax=m_true.max())
plt.colorbar(im, ax=ax, label='Slip (m)')
ax.invert_yaxis()
ax.set_xlabel('Along-strike patch'); ax.set_ylabel('Down-dip patch')
ax.set_title('True slip distribution — Gaussian asperity')
plt.tight_layout(); plt.show()


### 1b — Build the Green's function matrix and generate synthetic data

Fifteen GNSS stations are distributed perpendicular to the fault from −60 to +60 km.
We build **G** exactly as in Lab 10, then generate "observed" data as:

$$\mathbf{d}_{obs} = \mathbf{G}\mathbf{m}_{true} + \boldsymbol{\epsilon}$$

where $\boldsymbol{\epsilon}$ is Gaussian noise with standard deviation $\sigma$.
We start with $\sigma = 0$ (noiseless) and later add noise.


In [ ]:
# GNSS station positions (perpendicular to N-S striking fault)
N_STA = 15
x_sta = np.linspace(-60e3, 60e3, N_STA)
obs   = np.column_stack([x_sta, np.zeros(N_STA), np.zeros(N_STA)])

# Build G: East and North components interleaved (mm per m of slip)
unit_slip = np.array([np.cos(np.radians(180)), -np.sin(np.radians(180)), 0.])
G = np.zeros((N_STA * 2, N_PATCHES))
for k in range(N_PATCHES):
    tri_k  = tris_all[2*k:2*k+2]
    slip_k = np.tile(unit_slip, (2, 1))
    d_k    = hs.disp_free(obs, tri_k, slip_k, nu=NU)
    G[0::2, k] = d_k[:, 0] * 1000   # East  mm/m
    G[1::2, k] = d_k[:, 1] * 1000   # North mm/m

M_OBS = G.shape[0]
print(f"G shape: {G.shape}  ({M_OBS} obs × {N_PATCHES} patches)")
print(f"Condition number of G: {np.linalg.cond(G):.1e}")

# True (noiseless) data
d_true = G @ m_true

# Plot the noiseless data
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
x_km = x_sta / 1e3
for ax, comp, idx, label in zip(axes, ['East','North'], [0,1],
                                 ['E displacement (mm)','N displacement (mm)']):
    ax.plot(x_km, d_true[idx::2], 'k-', lw=2, label='True (noiseless)')
    ax.axvline(0, color='0.5', ls='--', lw=1, label='Fault trace')
    ax.set_xlabel('Distance from fault (km)'); ax.set_ylabel(label)
    ax.set_title(f'{comp} component — noiseless forward model')
    ax.legend(fontsize=9)
plt.suptitle('Synthetic GNSS data from the Gaussian asperity', fontsize=11)
plt.tight_layout(); plt.show()

print(f"\nData range: {d_true.min():.1f} to {d_true.max():.1f} mm")


### 1c — Noiseless inversion and its L-curve

First, try to recover the slip from **noiseless data** using ordinary least squares
(no regularization). Because $G$ is wide ($M < N$), we use the minimum-norm solution.
Then plot the L-curve.


In [ ]:
def solve_damped(G, d, lam, L=None):
    """
    Damped or smoothed least-squares solution.
    If L is None: Tikhonov damping (||m||^2).
    If L is given: smoothing (||Lm||^2).
    Returns m, predicted data, data misfit, model norm.
    """
    reg = L.T @ L if L is not None else np.eye(G.shape[1])
    A   = G.T @ G + lam**2 * reg
    m   = np.linalg.solve(A, G.T @ d)
    d_pred  = G @ m
    misfit  = np.linalg.norm(d - d_pred)
    mnorm   = np.linalg.norm(L @ m if L is not None else m)
    return m, d_pred, misfit, mnorm


def l_curve(G, d, lambdas, L=None):
    misfits, norms, solutions = [], [], []
    for lam in lambdas:
        m, _, misfit, mnorm = solve_damped(G, d, lam, L)
        misfits.append(misfit); norms.append(mnorm); solutions.append(m)
    return np.array(misfits), np.array(norms), solutions


# ── Noiseless inversion ───────────────────────────────────────────────────
# Very small lambda ≈ minimum-norm least squares
lam_tiny = 1e-6
m_noiseless, d_pred_nl, misfit_nl, _ = solve_damped(G, d_true, lam_tiny)

print("Noiseless inversion (lambda ~ 0):")
print(f"  Data misfit: {misfit_nl:.4f} mm")
print(f"  Recovered max slip: {m_noiseless.max():.2f} m  (true: {m_true.max():.2f} m)")

# L-curve for noiseless data
lambdas = np.logspace(-6, 2, 60)
mf_nl, nm_nl, _ = l_curve(G, d_true, lambdas)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(mf_nl, nm_nl, c=np.log10(lambdas), cmap='plasma', s=40)
plt.colorbar(sc, ax=ax, label='log10(lambda)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Data misfit ||Gm - d|| (mm)')
ax.set_ylabel('Model norm ||m|| (m)')
ax.set_title('L-curve — NOISELESS data\n(no corner)')

ax = axes[1]
m_nl_grid = m_noiseless.reshape(N_ALONG, N_DOWN)
m_true_grid = m_true.reshape(N_ALONG, N_DOWN)
x = np.arange(N_ALONG+1); y = np.arange(N_DOWN+1)
vmax = max(m_true.max(), m_noiseless.max())
ax.pcolormesh(x, y, m_true_grid.T, cmap='hot_r', vmin=0, vmax=vmax, alpha=0.5)
im = ax.pcolormesh(x, y, m_nl_grid.T, cmap='Blues', vmin=0, vmax=vmax, alpha=0.6)
plt.colorbar(im, ax=ax, label='Slip (m)')
ax.invert_yaxis()
ax.set_title(f'Recovered slip (noiseless, lambda={lam_tiny:.0e})\n'
             f'max={m_noiseless.max():.2f} m  (true: {m_true.max():.2f} m)')
ax.set_xlabel('Along-strike patch'); ax.set_ylabel('Down-dip patch')

plt.suptitle('Noiseless inversion', fontsize=11)
plt.tight_layout(); plt.show()


### 1d — Add noise and watch the solution explode

Now add realistic Gaussian noise ($\sigma = 2$ mm) and solve without regularization.


In [ ]:
np.random.seed(42)
SIGMA = 2.0   # mm — realistic GNSS daily position noise

d_noisy = d_true + np.random.normal(0, SIGMA, M_OBS)

# Attempt direct inversion (no regularization)
m_direct, d_pred_dir, misfit_dir, _ = solve_damped(G, d_noisy, lam_tiny)

print(f"Direct inversion with sigma={SIGMA} mm noise (lambda={lam_tiny:.0e}):")
print(f"  Data misfit: {misfit_dir:.2f} mm")
print(f"  Recovered max |slip|: {np.abs(m_direct).max():.0f} m  (true: {m_true.max():.2f} m)")

# L-curve for noisy data
mf_noisy, nm_noisy, _ = l_curve(G, d_noisy, lambdas)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Compare L-curves
ax = axes[0]
ax.loglog(mf_nl,    nm_nl,    'steelblue', lw=2, label='Noiseless')
ax.loglog(mf_noisy, nm_noisy, 'firebrick', lw=2, label=f'Noisy (sigma={SIGMA} mm)')
ax.axhline(np.linalg.norm(m_true), color='k', ls=':', lw=1, label='||m_true||')
ax.set_xlabel('Data misfit (mm)'); ax.set_ylabel('Model norm (m)')
ax.set_title('L-curves: noiseless vs noisy')
ax.legend(fontsize=9)

# Noisy data vs true
ax = axes[1]
x_km = x_sta / 1e3
ax.plot(x_km, d_true[0::2],  'k-',  lw=2, label='True')
ax.plot(x_km, d_noisy[0::2], 'r.',  ms=8, label=f'Noisy (sigma={SIGMA}mm)')
ax.set_xlabel('Distance from fault (km)'); ax.set_ylabel('East displacement (mm)')
ax.set_title('True vs noisy observations'); ax.legend(fontsize=9)

# Blown-up solution
ax = axes[2]
x_patch = np.arange(N_PATCHES)
ax.bar(x_patch - 0.2, m_true,   width=0.4, color='steelblue', label='True slip')
ax.bar(x_patch + 0.2, m_direct, width=0.4, color='firebrick',
       label=f'Recovered (no reg.)', alpha=0.7)
ax.set_xlabel('Patch index'); ax.set_ylabel('Slip (m)')
ax.set_title('Direct inversion explodes!')
ax.legend(fontsize=9)

plt.suptitle(f'Effect of noise (sigma={SIGMA} mm) without regularization', fontsize=11)
plt.tight_layout(); plt.show()


> **Part I questions:**
> 1. The L-curve for noiseless data has no clear corner — it just slopes continuously. Why? What is the L-curve actually identifying when it *does* have a corner?
> 2. The direct inversion with noise produces slip values orders of magnitude larger than the true answer. Why does a small amount of noise cause such a large error? What property of the matrix $\mathbf{G}$ determines how badly noise is amplified?
> 3. The condition number of $\mathbf{G}$ was printed above. What does a large condition number tell you about the inverse problem?
> 4. The noiseless direct inversion recovers the slip well. Does this mean the problem is easy without noise? What would happen if you used a different (incorrect) noise realization — would you still recover the true slip?


## Part II: Damping regularization

Tikhonov damping adds $\lambda^2\|\mathbf{m}\|^2$ to the objective — it prefers small slip.
The solution is:

$$\mathbf{m}_{damp} = (\mathbf{G}^T\mathbf{G} + \lambda^2\mathbf{I})^{-1}\mathbf{G}^T\mathbf{d}$$

Here you will explore how $\lambda$ controls the trade-off between fitting the data
and keeping the slip small. Because you know $\mathbf{m}_{true}$, you can directly
measure the **recovery error** $\|\mathbf{m}_{recovered} - \mathbf{m}_{true}\|$.


In [ ]:
# Sweep lambda values and compute recovery error
lambdas_fine = np.logspace(-4, 2, 80)
misfits_d, norms_d, errors_d, slips_d = [], [], [], []

for lam in lambdas_fine:
    m, d_pred, misfit, mnorm = solve_damped(G, d_noisy, lam, L=None)
    misfits_d.append(misfit)
    norms_d.append(mnorm)
    errors_d.append(np.linalg.norm(m - m_true))
    slips_d.append(m)

misfits_d = np.array(misfits_d)
norms_d   = np.array(norms_d)
errors_d  = np.array(errors_d)

# Find lambda that minimises recovery error (oracle — only possible because we know truth)
best_idx  = np.argmin(errors_d)
lam_best  = lambdas_fine[best_idx]
m_best_d  = slips_d[best_idx]

print(f"Lambda minimising recovery error: {lam_best:.4f}")
print(f"Min recovery error: {errors_d[best_idx]:.3f} m")
print(f"Recovered peak slip: {m_best_d.max():.2f} m  (true: {m_true.max():.2f} m)")

# Three panels: L-curve, error curve, comparison at best lambda
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
sc = ax.scatter(misfits_d, norms_d, c=np.log10(lambdas_fine),
                cmap='plasma', s=30)
plt.colorbar(sc, ax=ax, label='log10(lambda)')
ax.scatter(misfits_d[best_idx], norms_d[best_idx],
           s=150, c='k', marker='*', zorder=5, label=f'Best lambda={lam_best:.3f}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Data misfit (mm)'); ax.set_ylabel('Model norm (m)')
ax.set_title('L-curve — damping'); ax.legend(fontsize=9)

ax = axes[1]
ax.loglog(lambdas_fine, errors_d, 'steelblue', lw=2)
ax.axvline(lam_best, color='k', ls='--', lw=1.5,
           label=f'Best lambda={lam_best:.3f}')
ax.set_xlabel('lambda'); ax.set_ylabel('Recovery error ||m - m_true|| (m)')
ax.set_title('Recovery error vs lambda\n(oracle: only possible with known truth)')
ax.legend(fontsize=9)

ax = axes[2]
x_patch = np.arange(N_PATCHES)
ax.bar(x_patch - 0.2, m_true,   width=0.4, color='steelblue', label='True slip')
ax.bar(x_patch + 0.2, m_best_d, width=0.4, color='firebrick',
       alpha=0.8, label=f'Damped (lambda={lam_best:.3f})')
ax.set_xlabel('Patch index'); ax.set_ylabel('Slip (m)')
ax.set_title('True vs damped solution (best lambda)')
ax.legend(fontsize=9)

plt.suptitle(f'Damping regularization  (sigma={SIGMA} mm)', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Explore three lambda values: too small, about right, too large
lam_vals = [0.01, lam_best, 2.0]
labels   = [f'lambda={l:.3f}' for l in lam_vals]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lam, label in zip(axes, lam_vals, labels):
    m, _, _, _ = solve_damped(G, d_noisy, lam, L=None)
    err = np.linalg.norm(m - m_true)
    x = np.arange(N_PATCHES)
    ax.bar(x - 0.2, m_true, width=0.4, color='steelblue', label='True')
    ax.bar(x + 0.2, m,      width=0.4, color='firebrick',
           alpha=0.8, label=f'Damped (err={err:.2f} m)')
    ax.set_ylim(-1, m_true.max()*1.5)
    ax.set_xlabel('Patch index'); ax.set_ylabel('Slip (m)')
    ax.set_title(label); ax.legend(fontsize=8)

plt.suptitle('Effect of damping parameter on recovered slip', fontsize=11)
plt.tight_layout(); plt.show()

print("Notice: too small -> noisy; too large -> everything squeezed toward zero")


> **Part II questions:**
> 1. The "best" $\lambda$ was found by minimising the recovery error against the known true solution. In a real inversion you don't have the true answer. What method can you use instead to choose $\lambda$?
> 2. Look at the recovered peak slip at the best $\lambda$. Is it larger or smaller than the true peak? Why does damping systematically bias the amplitude in this direction?
> 3. At very large $\lambda$, all patch slips are driven toward zero. What does this correspond to physically — what prior belief about the earthquake are you imposing?
> 4. The recovery error curve has a minimum. Why does the error *increase* again at very small $\lambda$ (not just stay low once the regularization is removed)?


## Part III: Smoothing regularization

Smoothing penalises **spatial roughness** rather than model size:

$$\min_{\mathbf{m}}\; \|\mathbf{G}\mathbf{m} - \mathbf{d}\|^2 + \lambda^2\|\mathbf{L}\mathbf{m}\|^2$$

where $\mathbf{L}$ is the finite-difference Laplacian operator on the patch grid.
The solution is:

$$\mathbf{m}_{smooth} = (\mathbf{G}^T\mathbf{G} + \lambda^2\mathbf{L}^T\mathbf{L})^{-1}\mathbf{G}^T\mathbf{d}$$

Adjacent patches are encouraged to have similar slip — geologically plausible —
but sharp features get blurred.


In [ ]:
# Sweep lambda for smoothing
misfits_s, norms_s, errors_s, slips_s = [], [], [], []

for lam in lambdas_fine:
    m, d_pred, misfit, mnorm = solve_damped(G, d_noisy, lam, L=L_lap)
    misfits_s.append(misfit)
    norms_s.append(mnorm)
    errors_s.append(np.linalg.norm(m - m_true))
    slips_s.append(m)

misfits_s = np.array(misfits_s)
norms_s   = np.array(norms_s)
errors_s  = np.array(errors_s)

best_idx_s = np.argmin(errors_s)
lam_best_s = lambdas_fine[best_idx_s]
m_best_s   = slips_s[best_idx_s]

print(f"Smoothing — best lambda: {lam_best_s:.4f}")
print(f"  Min recovery error: {errors_s[best_idx_s]:.3f} m")
print(f"  Recovered peak slip: {m_best_s.max():.2f} m  (true: {m_true.max():.2f} m)")
print(f"\nDamping  — best recovery error: {errors_d.min():.3f} m")
print(f"Smoothing — best recovery error: {errors_s.min():.3f} m")

# Compare damping vs smoothing
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.loglog(misfits_d, norms_d,   'steelblue', lw=2, label='Damping')
ax.loglog(misfits_s, norms_s,   'firebrick', lw=2, label='Smoothing')
ax.set_xlabel('Data misfit (mm)'); ax.set_ylabel('Regularization norm')
ax.set_title('L-curves: damping vs smoothing'); ax.legend(fontsize=9)

ax = axes[1]
ax.loglog(lambdas_fine, errors_d, 'steelblue', lw=2, label='Damping')
ax.loglog(lambdas_fine, errors_s, 'firebrick', lw=2, label='Smoothing')
ax.axvline(lam_best,   color='steelblue', ls='--', lw=1)
ax.axvline(lam_best_s, color='firebrick', ls='--', lw=1)
ax.set_xlabel('lambda'); ax.set_ylabel('Recovery error ||m - m_true|| (m)')
ax.set_title('Recovery error: damping vs smoothing'); ax.legend(fontsize=9)

ax = axes[2]
x_patch = np.arange(N_PATCHES)
ax.bar(x_patch - 0.3, m_true,   width=0.25, color='0.4',      label='True')
ax.bar(x_patch,       m_best_d, width=0.25, color='steelblue',
       alpha=0.85, label=f'Damped (lam={lam_best:.3f})')
ax.bar(x_patch + 0.3, m_best_s, width=0.25, color='firebrick',
       alpha=0.85, label=f'Smooth (lam={lam_best_s:.3f})')
ax.set_xlabel('Patch index'); ax.set_ylabel('Slip (m)')
ax.set_title('True vs best-lambda solutions')
ax.legend(fontsize=8)

plt.suptitle(f'Damping vs smoothing (sigma={SIGMA} mm)', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Visualise slip maps on the fault plane for both regularizations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = m_true.max()

for ax, m, title in zip(axes,
        [m_true, m_best_d, m_best_s],
        ['True slip', f'Damped (lam={lam_best:.3f})',
         f'Smoothed (lam={lam_best_s:.3f})']):
    im = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                       m.reshape(N_ALONG, N_DOWN).T,
                       cmap='hot_r', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Slip (m)')
    ax.invert_yaxis()
    ax.set_xlabel('Along-strike'); ax.set_ylabel('Down-dip')
    ax.set_title(title)

plt.suptitle('Slip distribution on fault plane', fontsize=11)
plt.tight_layout(); plt.show()


> **Part III questions:**
> 1. Which regularization type — damping or smoothing — gives a lower recovery error at its best $\lambda$? Does the better-performing method also give a more geologically plausible slip pattern?
> 2. Look at the fault-plane maps. The smoothed solution has spatial coherence but the asperity is blurred. The damped solution has the correct general shape but with amplitude bias. Which would you trust more in a real inversion, and why?
> 3. The Laplacian $\mathbf{L}$ penalises differences between adjacent patches. What geological prior does this encode? Is it always a good prior for earthquake slip?
> 4. Both methods systematically underestimate the peak slip. Explain why this bias is unavoidable for any regularized inversion, regardless of how well you choose $\lambda$.


## Part IV: The L-curve

In a real inversion you don't know $\mathbf{m}_{true}$, so you can't minimise the
recovery error directly. The **L-curve** is the standard practical tool: plot model
norm (or roughness) against data misfit as $\lambda$ varies, and look for the corner.

Here you can compare the L-curve-selected $\lambda$ with the oracle best $\lambda$
from Parts II and III — how close does the L-curve method come to the true optimum?


In [ ]:
def find_l_curve_corner(misfits, norms):
    """
    Approximate the L-curve corner using maximum curvature in log-log space.
    Returns the index of the corner point.
    """
    log_m = np.log(misfits); log_n = np.log(norms)
    # Numerical second derivative of log_n with respect to log_m
    dm = np.gradient(log_m); dn = np.gradient(log_n)
    d2n = np.gradient(dn / (dm + 1e-12))
    # Corner = maximum curvature (most negative d^2n/dm^2 in the L shape)
    curvature = -d2n / (1 + (dn/(dm+1e-12))**2)**1.5
    return np.argmax(curvature)


# L-curve corners for both regularizations
corner_d = find_l_curve_corner(misfits_d, norms_d)
corner_s = find_l_curve_corner(misfits_s, norms_s)
lam_lcurve_d = lambdas_fine[corner_d]
lam_lcurve_s = lambdas_fine[corner_s]

print(f"Damping:   L-curve lambda={lam_lcurve_d:.4f},  oracle best={lam_best:.4f}")
print(f"Smoothing: L-curve lambda={lam_lcurve_s:.4f},  oracle best={lam_best_s:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, mf, nm, corner, lam_lc, lam_or, title in zip(
        axes,
        [misfits_d, misfits_s], [norms_d, norms_s],
        [corner_d, corner_s],
        [lam_lcurve_d, lam_lcurve_s],
        [lam_best, lam_best_s],
        ['Damping', 'Smoothing']):

    sc = ax.scatter(mf, nm, c=np.log10(lambdas_fine), cmap='plasma', s=30)
    plt.colorbar(sc, ax=ax, label='log10(lambda)')
    ax.scatter(mf[corner], nm[corner], s=200, c='k', marker='*',
               zorder=6, label=f'L-curve corner: lambda={lam_lc:.4f}')
    # Mark oracle
    idx_or = np.argmin(np.abs(lambdas_fine - lam_or))
    ax.scatter(mf[idx_or], nm[idx_or], s=150, c='lime', marker='D',
               zorder=6, edgecolors='k', label=f'Oracle best: lambda={lam_or:.4f}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Data misfit (mm)'); ax.set_ylabel('Regularization norm')
    ax.set_title(f'L-curve — {title}'); ax.legend(fontsize=8)

plt.suptitle('L-curve corner vs oracle-best lambda', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Compare L-curve solution vs oracle solution for smoothing
m_lcurve_d, _, _, _ = solve_damped(G, d_noisy, lam_lcurve_d, L=None)
m_lcurve_s, _, _, _ = solve_damped(G, d_noisy, lam_lcurve_s, L=L_lap)

err_lcurve_d = np.linalg.norm(m_lcurve_d - m_true)
err_lcurve_s = np.linalg.norm(m_lcurve_s - m_true)

print(f"Damping:")
print(f"  L-curve lambda={lam_lcurve_d:.4f}  -> recovery error = {err_lcurve_d:.3f} m")
print(f"  Oracle  lambda={lam_best:.4f}  -> recovery error = {errors_d.min():.3f} m")
print(f"Smoothing:")
print(f"  L-curve lambda={lam_lcurve_s:.4f}  -> recovery error = {err_lcurve_s:.3f} m")
print(f"  Oracle  lambda={lam_best_s:.4f}  -> recovery error = {errors_s.min():.3f} m")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmax = m_true.max()

for ax, m, title in zip(axes,
        [m_true, m_lcurve_d, m_lcurve_s],
        ['True slip',
         f'Damped — L-curve lambda={lam_lcurve_d:.3f}\n(err={err_lcurve_d:.2f} m)',
         f'Smoothed — L-curve lambda={lam_lcurve_s:.3f}\n(err={err_lcurve_s:.2f} m)']):
    im = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                       m.reshape(N_ALONG, N_DOWN).T,
                       cmap='hot_r', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Slip (m)')
    ax.invert_yaxis()
    ax.set_xlabel('Along-strike'); ax.set_ylabel('Down-dip')
    ax.set_title(title)

plt.suptitle('L-curve-selected solutions vs true slip', fontsize=11)
plt.tight_layout(); plt.show()


> **Part IV questions:**
> 1. How close is the L-curve $\lambda$ to the oracle best $\lambda$ for damping? For smoothing? Is the L-curve a reliable guide in both cases?
> 2. Re-run the cells above with a different noise level — try $\sigma = 0.5$ mm (low noise) and $\sigma = 5$ mm (high noise). How does the L-curve corner shift? Does the recovery error at the L-curve corner change?
> 3. The L-curve corner is found by maximum curvature in log-log space. What does it mean geometrically when the L-curve has no clear corner — like in the noiseless case from Part I?
> 4. In the Cascadia lab (Lab 12) you will use the L-curve with real data and have no oracle to compare against. Based on what you learned here, how confident should you be that the L-curve corner gives you the "correct" regularization?


## Part V: The resolution matrix

The resolution matrix $\mathbf{R}$ tells you what the inversion actually measures.
If $\mathbf{m}_{true}$ is the true slip, the recovered slip is:

$$\mathbf{m}_{recovered} = \mathbf{R}\,\mathbf{m}_{true} + \text{noise terms}$$

If $\mathbf{R} = \mathbf{I}$, recovery is perfect. In practice $\mathbf{R} \neq \mathbf{I}$:
the recovered slip at each patch is a **weighted average of nearby true slip values**,
with the weights given by the corresponding row of $\mathbf{R}$.

Because you know $\mathbf{m}_{true}$ here, you can verify this directly: compare
$\mathbf{R}\mathbf{m}_{true}$ (the "predicted recovery") to the actual inversion output.


In [ ]:
# Compute R for smoothing at the L-curve lambda
GtG = G.T @ G
LtL = L_lap.T @ L_lap
A   = GtG + lam_lcurve_s**2 * LtL
A_inv = np.linalg.inv(A)
R   = A_inv @ GtG

R_diag = np.diag(R)

print(f"Resolution matrix: shape {R.shape}")
print(f"Diagonal range: {R_diag.min():.3f} to {R_diag.max():.3f}")
print(f"Mean self-resolution: {R_diag.mean():.3f}")
print(f"\nPatches with R > 0.8: {(R_diag > 0.8).sum()}")
print(f"Patches with R < 0.3: {(R_diag < 0.3).sum()}")

# Verify: R @ m_true should match the recovered slip (up to noise)
m_predicted_by_R = R @ m_true
print(f"\n||R m_true - m_lcurve||: {np.linalg.norm(m_predicted_by_R - m_lcurve_s):.4f} m")
print(f"(Should be small — R m_true predicts the deterministic part of recovery)")


In [ ]:
# Plot resolution diagonal on the fault
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
im = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                   R_diag.reshape(N_ALONG, N_DOWN).T,
                   cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Self-resolution (diagonal of R)')
ax.invert_yaxis()
ax.set_xlabel('Along-strike'); ax.set_ylabel('Down-dip')
ax.set_title(f'Resolution diagonal\n(lambda={lam_lcurve_s:.3f})')

# True, R@m_true, and recovered
ax = axes[1]
x = np.arange(N_PATCHES)
ax.bar(x - 0.3, m_true,               width=0.25, color='0.4',       label='True m')
ax.bar(x,       m_predicted_by_R,     width=0.25, color='goldenrod',  label='R @ m_true')
ax.bar(x + 0.3, m_lcurve_s,           width=0.25, color='firebrick',
       alpha=0.8, label='Recovered')
ax.set_xlabel('Patch index'); ax.set_ylabel('Slip (m)')
ax.set_title('True vs R@m_true vs recovered\n(R@m_true is the blur, not the noise)')
ax.legend(fontsize=8)

# Resolution kernel for the asperity patch
asperity_patch = np.argmax(m_true)   # highest slip patch
kernel = R[asperity_patch, :]

ax = axes[2]
im2 = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                    kernel.reshape(N_ALONG, N_DOWN).T,
                    cmap='RdBu_r', vmin=-np.abs(kernel).max(), vmax=np.abs(kernel).max())
plt.colorbar(im2, ax=ax, label='Kernel weight')
ax.invert_yaxis()
ax.set_xlabel('Along-strike'); ax.set_ylabel('Down-dip')
ax.set_title(f'Resolution kernel for asperity patch {asperity_patch}\n'
             f'(self-resolution = {R_diag[asperity_patch]:.2f})')

plt.suptitle(f'Resolution matrix — smoothing, lambda={lam_lcurve_s:.3f}', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# How does resolution change with lambda?
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
lam_explore = [0.01, lam_lcurve_s, 1.0]

for col, lam in enumerate(lam_explore):
    A_l   = GtG + lam**2 * LtL
    R_l   = np.linalg.inv(A_l) @ GtG
    diag  = np.diag(R_l)
    kern  = R_l[asperity_patch, :]

    ax = axes[0, col]
    im = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                       diag.reshape(N_ALONG, N_DOWN).T,
                       cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.invert_yaxis()
    ax.set_title(f'Resolution diagonal\nlambda={lam}')
    ax.set_xlabel('Along-strike'); ax.set_ylabel('Down-dip')

    ax = axes[1, col]
    vk = np.abs(kern).max()
    im2 = ax.pcolormesh(np.arange(N_ALONG+1), np.arange(N_DOWN+1),
                        kern.reshape(N_ALONG, N_DOWN).T,
                        cmap='RdBu_r', vmin=-vk, vmax=vk)
    plt.colorbar(im2, ax=ax)
    ax.invert_yaxis()
    ax.set_title(f'Asperity kernel\nlambda={lam}, R_self={diag[asperity_patch]:.2f}')
    ax.set_xlabel('Along-strike'); ax.set_ylabel('Down-dip')

plt.suptitle('Resolution vs lambda — how regularization blurs the solution', fontsize=11)
plt.tight_layout(); plt.show()


> **Part V questions:**
> 1. The resolution diagonal ranges from 0 to 1. Which patches are best resolved? Is there a spatial pattern — do certain parts of the fault consistently have higher resolution?
> 2. The equation $\mathbf{m}_{recovered} \approx \mathbf{R}\mathbf{m}_{true}$ says the inversion blurs the true slip. How wide is the resolution kernel for the asperity patch? Does this explain the discrepancy between the true peak slip and the recovered peak slip you observed in Parts II and III?
> 3. As $\lambda$ increases, the resolution kernels get broader. Explain why — what is larger regularization doing to the blurring?
> 4. In the Cascadia lab you will compute $\mathbf{R}$ for the SSE inversion. Based on what you learned here, predict: which patches in the Cascadia inversion will have the highest resolution? Which will have the lowest? Why?
> 5. The resolution matrix was derived assuming linear (no NNLS) regularization. The Cascadia lab uses NNLS (non-negative least squares). How might the non-negativity constraint change the effective resolution compared to what $\mathbf{R}$ predicts?


## Synthesis

> **1. The fundamental problem**  
> In your own words, explain in 3–5 sentences why noise makes the inverse problem hard even when the forward problem is trivial. Your answer should reference the condition number of $\mathbf{G}$, the amplification of noise, and why regularization helps.

> **2. Damping vs smoothing**  
> Compare the two regularization strategies. For a real earthquake — where you expect slip to be spatially coherent and concentrated on an asperity — which is more appropriate and why? Are there situations where damping would be preferable?

> **3. The L-curve as a practical tool**  
> You had access to the true answer in this lab. In the Cascadia SSE inversion (Lab 12) you do not. Describe the procedure you will use to choose $\lambda$, and explain what the corner of the L-curve is identifying physically.

> **4. What the resolution matrix tells you**  
> Suppose the Cascadia inversion recovers 3 cm of peak slip on a patch at 30 km depth. The resolution diagonal for that patch is 0.3. What does this tell you about the true slip? If the asperity patch had a resolution of 0.9 instead, how would your interpretation change?


## Summary

- **Noise amplification** is the core difficulty of inverse problems. Even small noise can produce solution errors orders of magnitude larger than the signal when $\mathbf{G}$ is ill-conditioned. The condition number of $\mathbf{G}$ quantifies this sensitivity.

- The **L-curve has no corner for noiseless data** — the corner only emerges because noise creates a noise floor. This means the L-curve is identifying the regularization that balances fitting the signal against amplifying the noise, not simply minimising misfit.

- **Damping** ($\lambda^2\|\mathbf{m}\|^2$) prefers small slip and stabilises the solution but systematically underestimates amplitude. **Smoothing** ($\lambda^2\|\mathbf{L}\mathbf{m}\|^2$) produces spatially coherent results but blurs sharp features. Neither perfectly recovers the true slip — the bias is unavoidable.

- The **L-curve corner** approximates the optimal $\lambda$ without knowing the true answer. It tends to be close to the oracle best $\lambda$ for well-behaved problems, but should be interpreted as a guide rather than a guarantee.

- The **resolution matrix** $\mathbf{R}$ quantifies the blurring introduced by regularization. The recovered slip is $\mathbf{m}_{recovered} \approx \mathbf{R}\mathbf{m}_{true}$: a spatially smoothed version of the truth, not the truth itself. The resolution diagonal tells you how well each patch is constrained; the resolution kernel tells you what spatial average it actually measures.

- Larger $\lambda$ → broader kernels → more blurring → lower self-resolution. There is a fundamental trade-off between stability (large $\lambda$) and resolution (small $\lambda$).
